# Step 6 · Initial error analysis

**Status:** Initial per-word diagnostics completed; targeted improvements and full period study pending.

Inspect both saved score sets without rerunning models. This notebook is exploratory: inspecting development errors does not create a new independent test set.

Start with a fresh runtime and execute cells in order. Read each section before running it. The original combined notebook and its outputs are retained as historical evidence.

## 1. Connect to the project
CPU is sufficient unless you explicitly enable new model inference. This uses the existing Drive project; it does not delete or recopy it.

In [ ]:
from pathlib import Path
import sys, os
try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd().resolve()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
assert (PROJECT/'src/semeval_task3/data.py').is_file(), f'Edit PROJECT to your project root: {PROJECT}'
assert (PROJECT/'notebooks/project_utils.py').is_file(), 'Use the organized project, including notebooks/project_utils.py.'
sys.path.insert(0, str(PROJECT/'src'))
sys.path.insert(0, str(PROJECT/'notebooks'))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
from project_utils import (run_checked, new_run, save_json, read_scores, save_scores,
                           frozen, standardized_by_word, record_environment, save_evaluation, digest)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, word_bootstrap, read_submission
print('Project:', PROJECT)
print('Existing files stay in place. New outputs use unique directories under outputs/runs/.')

## 2. Load saved scores and diagnostic dependencies

In [ ]:
run_checked([sys.executable,'-m','pip','install','--quiet','numpy','scikit-learn'],cwd=PROJECT)
import numpy as np
from sklearn.metrics import roc_auc_score
dev = load_dataset(PROJECT/'work/data/dev',with_gold=True)
qwen = read_scores(PROJECT/'outputs/qwen3_8b_zs_scores.json',dev)
e5 = read_scores(PROJECT/'outputs/e5_raw_scores.json',dev)
RUN_DIR = new_run(PROJECT,'step06_analysis')
record_environment(PROJECT,RUN_DIR/'manifest.json',purpose='Exploratory diagnostics on historical frozen scores',
                   qwen_score_sha256=digest(PROJECT/'outputs/qwen3_8b_zs_scores.json'),e5_score_sha256=digest(PROJECT/'outputs/e5_raw_scores.json'))

## 3. Per-word ranking and Yes preference
 AUC measures ranking, not official F1. `Qwen Yes %` uses log-odds > 0 and is distinct from the tuned out-of-fold decision threshold. AUC is unavailable when a group has only one class.

In [ ]:
rows = []
print(f"{'word':12} {'n':>5} {'positive%':>10} {'Yes%':>8} {'AUC Qwen':>10} {'AUC E5':>10}")
for group in sorted({u.group for u in dev.usages}):
    keys = [u.key for u in dev.usages if u.group==group]
    y = [dev.gold[k] for k in keys]
    aq = float(roc_auc_score(y,[qwen[k] for k in keys])) if len(set(y))==2 else None
    ae = float(roc_auc_score(y,[e5[k] for k in keys])) if len(set(y))==2 else None
    row = dict(language=group[0],word=group[1],n=len(keys),positive_rate=sum(y)/len(y),
               qwen_yes_rate=sum(qwen[k]>0 for k in keys)/len(keys),auc_qwen=aq,auc_e5=ae)
    rows.append(row)
    print(group[1],row)
save_json(RUN_DIR/'per_word_diagnostics.json',rows)

## 4. Highest-scored labeled examples
 Reproduce the original focus on `fröken` and `kvinna`. These are high-scored negatives/positives; the labels below describe gold classes, not a complete confusion matrix at the tuned threshold. Keep the target in the excerpt.

In [ ]:
examples = []
for word in ['fröken','kvinna']:
    selected = [u for u in dev.usages if u.word==word and u.language=='SV']
    if not selected:
        continue
    print('\n',word,'—',selected[0].definition)
    for label,limit in [(0,4),(1,2)]:
        chosen = sorted((u for u in selected if dev.gold[u.key]==label),key=lambda u:-qwen[u.key])[:limit]
        for u in chosen:
            excerpt = u.text[max(0,u.start-100):u.start]+'<t>'+u.text[u.start:u.end]+'</t>'+u.text[u.end:u.end+160]
            row = dict(language=u.language,word=u.word,sentence_id=u.sentence_id,year=u.year,
                       gold=label,qwen_score=qwen[u.key],definition=u.definition,excerpt=excerpt,
                       error_category=None,reviewer_note='')
            examples.append(row)
            print(f"gold={label} score={qwen[u.key]:+.2f} year={u.year}: {excerpt}")
save_json(RUN_DIR/'examples_for_manual_review.json',examples)
print('Saved:',RUN_DIR)

## Still to do
 - Manually categorize a broader error sample: OCR, archaic language, metaphor, borderline sense, annotation disagreement.
 - Test a small number of targeted changes; use paired word-level differences and acknowledge repeated development tuning.
 - Analyze period performance alongside counts and positive rates. Raw period F1 alone cannot isolate an age effect from changes in word mix or class balance.
 - Validate other languages when appropriate data becomes available.

 No annotations are changed by this notebook. The historical-age analysis and final report are not complete.